# Analysis Open Position and Portoflio

In [1]:
import pandas as pd 
import refinitiv.data as rd

In [2]:
rd.open_session()

<refinitiv.data.session.Definition object at 0x11ee21b50 {name='workspace'}>

## Import the csv portfolio current

In [4]:
df_port = pd.read_csv("/Users/phamthanh/Documents/ESSCA | MSc Finance & Data Analyst/14. ESG Portfolio/v1.5/Open Positions Mar 8 2026.csv")

In [5]:
df_port.head()

,Symbol,Description,Quantity,Currency,LastPrice,PricePaid,DayChange,ProfitLoss,MarketValue,ProfitLossPercentage
0,AD,Agilent Technologies Inc.,130000,USD,49.03,48.17,-0.32,"96,283.86",5.489299e+06,1.79
1,EIX,Edison International,100000,USD,71.76,73.85,0.54,"-179,993.97",6.180080e+06,-2.83
2,FDX,Fedex Corp,20000,USD,359.10,382.79,-14.25,"-408,043.75",6.185247e+06,-6.19
3,FTI,TechnipFMC plc,76000,USD,63.02,67.12,-1.42,"-268,354.65",4.124807e+06,-6.11
4,GRMN,Garmin Ltd,23700,USD,243.48,250.00,3.31,"-133,078.41",4.969621e+06,-2.61


In [7]:
companies = df_port["Description"].tolist()

In [9]:
resolved =[]
for name in companies:
    s = rd.discovery.search(
        query = name,
        view = rd.discovery.Views.ORGANISATIONS,
        top=5,
        select = "DocumentTitle,CommonName,RCSAssetCategoryLeaf,OAPermID"
    )

    if s is not None and len(s) > 0:
        row = s.iloc[0]
        resolved.append({
                "input_name": name,
                "matched_name": row.get("CommonName"),
                "display_name": row.get("DocumentTitle"),
                "permid": row.get("OAPermID")
        })
        
resolved_df = pd.DataFrame(resolved)
print(resolved_df)

                        input_name              matched_name  \
0        Agilent Technologies Inc.  Agilent Technologies Inc   
1             Edison International      Edison International   
2                       Fedex Corp                FedEx Corp   
3                   TechnipFMC plc            TechnipFMC PLC   
4                       Garmin Ltd                Garmin Ltd   
5                  Hershey Company                Hershey Co   
6             Howmet Aerospace Inc      Howmet Aerospace Inc   
7                    ARCELORMITTAL          ArcelorMittal SA   
8              Omnicom Group, Inc.         Omnicom Group Inc   
9                       PG&E Corp.                 PG&E Corp   
10     Southern Copper Corporation      Southern Copper Corp   
11  Texas Pacific Land Corporation   Texas Pacific Land Corp   
12                    Tapestry Inc              Tapestry Inc   

                                display_name      permid  
0   Agilent Technologies Inc, Public Company

In [10]:
ids = [str(x) for x in resolved_df["permid"].dropna().tolist()]

industry_df = rd.get_data(
    universe=ids,
    fields=[
        "TR.CommonName",
        "TR.TRBCEconomicSector",
        "TR.TRBCBusinessSector",
        "TR.TRBCIndustryGroup",
        "TR.TRBCIndustry"
    ]
)

print(industry_df)

    Instrument       Company Common Name TRBC Economic Sector Name  \
0   4295908720  Agilent Technologies Inc                Healthcare   
1   4295903910      Edison International                 Utilities   
2   4295912126                FedEx Corp               Industrials   
3   5051791783            TechnipFMC PLC                    Energy   
4   4295913863                Garmin Ltd                Technology   
5   4295904171                Hershey Co    Consumer Non-Cyclicals   
6   5001815447      Howmet Aerospace Inc               Industrials   
7   5000030092          ArcelorMittal SA           Basic Materials   
8   4295904652         Omnicom Group Inc        Consumer Cyclicals   
9   4295904675                 PG&E Corp                 Utilities   
10  4295904951      Southern Copper Corp           Basic Materials   
11  5079228126   Texas Pacific Land Corp                    Energy   
12  4295901506              Tapestry Inc        Consumer Cyclicals   

          TRBC Busi